# Codificacion Huffman

## En R

In [2]:
#library(collections)

# ── Nodo del árbol Huffman ──────────────────────────────────────────────────
nuevo_nodo <- function(caracter, frecuencia) {
  list(
    caracter   = caracter,
    frecuencia = frecuencia,
    izquierda  = NULL,
    derecha    = NULL
  )
}

# ── Construir árbol Huffman ─────────────────────────────────────────────────
construir_arbol <- function(texto) {
  chars      <- strsplit(texto, "")[[1]]
  frecuencias <- table(chars)

  # Min-heap simulado como lista ordenada
  heap <- lapply(names(frecuencias), function(c) {
    nuevo_nodo(c, as.integer(frecuencias[c]))
  })

  heap_push <- function(heap, nodo) {
    heap[[length(heap) + 1]] <- nodo
    heap[order(sapply(heap, `[[`, "frecuencia"))]
  }

  heap_pop <- function(heap) {
    list(nodo = heap[[1]], heap = heap[-1])
  }

  while (length(heap) > 1) {
    res_izq  <- heap_pop(heap);  izquierda <- res_izq$nodo;  heap <- res_izq$heap
    res_der  <- heap_pop(heap);  derecha   <- res_der$nodo;  heap <- res_der$heap

    combinado            <- nuevo_nodo(NULL, izquierda$frecuencia + derecha$frecuencia)
    combinado$izquierda  <- izquierda
    combinado$derecha    <- derecha

    heap <- heap_push(heap, combinado)
  }

  heap[[1]]
}

# ── Generar códigos ─────────────────────────────────────────────────────────
generar_codigos <- function(nodo, codigo = "", tabla = list()) {
  if (is.null(nodo)) return(tabla)

  if (!is.null(nodo$caracter)) {
    tabla[[nodo$caracter]] <- codigo
    return(tabla)
  }

  tabla <- generar_codigos(nodo$izquierda, paste0(codigo, "0"), tabla)
  tabla <- generar_codigos(nodo$derecha,   paste0(codigo, "1"), tabla)
  tabla
}

# ── Codificar texto ─────────────────────────────────────────────────────────
codificar <- function(texto, tabla) {
  chars <- strsplit(texto, "")[[1]]
  paste0(sapply(chars, function(c) tabla[[c]]), collapse = "")
}

# ── Decodificar bits ────────────────────────────────────────────────────────
decodificar <- function(bits, raiz) {
  resultado    <- character(0)
  nodo_actual  <- raiz
  bit_vec      <- strsplit(bits, "")[[1]]

  for (bit in bit_vec) {
    nodo_actual <- if (bit == "0") nodo_actual$izquierda else nodo_actual$derecha

    if (!is.null(nodo_actual$caracter)) {
      resultado   <- c(resultado, nodo_actual$caracter)
      nodo_actual <- raiz
    }
  }

  paste0(resultado, collapse = "")
}

# ── Leer archivo ────────────────────────────────────────────────────────────
ruta  <- "texto.txt"
texto <- readLines(ruta, encoding = "UTF-8", warn = FALSE)
texto <- paste0(texto, collapse = "\n")

# ── Construcción ────────────────────────────────────────────────────────────
raiz  <- construir_arbol(texto)

# ── Tabla Huffman ───────────────────────────────────────────────────────────
tabla <- generar_codigos(raiz)
cat("TABLA DE CÓDIGOS:\n")
for (k in names(tabla)) {
  cat(sprintf("%s : %s\n", deparse(k), tabla[[k]]))
}

# ── Codificación ────────────────────────────────────────────────────────────
bits <- codificar(texto, tabla)
cat("\nTEXTO CODIFICADO:\n", bits, "\n")

# ── Decodificación ──────────────────────────────────────────────────────────
texto_decodificado <- decodificar(bits, raiz)
cat("\nTEXTO DECODIFICADO:\n", texto_decodificado, "\n")

# ── Frecuencias ─────────────────────────────────────────────────────────────
chars       <- strsplit(texto, "")[[1]]
frecuencias <- table(chars)
cat("\nFRECUENCIAS:\n")
for (c in names(frecuencias)) {
  cat(sprintf("%s : %d\n", deparse(c), frecuencias[c]))
}

# ── Verificación ─────────────────────────────────────────────────────────────
if (texto == texto_decodificado) {
  cat("\nVerificación exitosa.\n")
} else {
  cat("\nError en decodificación.\n")
}

TABLA DE CÓDIGOS:
"f" : 000
"h" : 001
"m" : 010
"n" : 011
"o" : 100
"u" : 101
"d" : 1100
"l" : 1101
" " : 1110
"a" : 1111

TEXTO CODIFICADO:
 001100110111111110010101011110010011100011010000000101111011 

TEXTO DECODIFICADO:
 hola mundo huffman 

FRECUENCIAS:
" " : 2
"a" : 2
"d" : 1
"f" : 2
"h" : 2
"l" : 1
"m" : 2
"n" : 2
"o" : 2
"u" : 2

Verificación exitosa.


## En python

Uso de Gemini, prompt: Traduce el siguiente código en R a python

In [1]:
import heapq
from collections import Counter

# Nodo del árbol Huffman
class Nodo:
    def __init__(self, caracter, frecuencia):
        self.caracter = caracter
        self.frecuencia = frecuencia
        self.izquierda = None
        self.derecha = None

    def __lt__(self, otro):
        return self.frecuencia < otro.frecuencia


# Construir árbol Huffman
def construir_arbol(texto):
    frecuencias = Counter(texto)

    heap = []

    for caracter, frecuencia in frecuencias.items():
        nodo = Nodo(caracter, frecuencia)
        heapq.heappush(heap, nodo)

    while len(heap) > 1:
        izquierda = heapq.heappop(heap)
        derecha = heapq.heappop(heap)

        combinado = Nodo(None, izquierda.frecuencia + derecha.frecuencia)
        combinado.izquierda = izquierda
        combinado.derecha = derecha

        heapq.heappush(heap, combinado)

    return heap[0]


# Generar códigos
def generar_codigos(nodo, codigo="", tabla=None):
    if tabla is None:
        tabla = {}

    if nodo is None:
        return tabla

    if nodo.caracter is not None:
        tabla[nodo.caracter] = codigo

    generar_codigos(nodo.izquierda, codigo + "0", tabla)
    generar_codigos(nodo.derecha, codigo + "1", tabla)

    return tabla


# Codificar texto
def codificar(texto, tabla):
    resultado = ""

    for caracter in texto:
        resultado += tabla[caracter]

    return resultado


# Decodificar bits
def decodificar(bits, raiz):
    resultado = ""
    nodo_actual = raiz

    for bit in bits:
        if bit == "0":
            nodo_actual = nodo_actual.izquierda
        else:
            nodo_actual = nodo_actual.derecha

        if nodo_actual.caracter is not None:
            resultado += nodo_actual.caracter
            nodo_actual = raiz

    return resultado


# Leer archivo
ruta = "texto.txt"

with open(ruta, "r", encoding="utf-8") as archivo:
    texto = archivo.read()

# Construcción
raiz = construir_arbol(texto)

# Tabla Huffman
tabla = generar_codigos(raiz)

print("TABLA DE CÓDIGOS:")
for k, v in tabla.items():
    print(f"{repr(k)} : {v}")

# Codificación
bits = codificar(texto, tabla)

print("\nTEXTO CODIFICADO:")
print(bits)

# Decodificación
texto_decodificado = decodificar(bits, raiz)

print("\nTEXTO DECODIFICADO:")
print(texto_decodificado)

# Contador de frecuencias
frecuencias = Counter(texto)

print("FRECUENCIAS:")

for caracter, frecuencia in frecuencias.items():
    print(f"{repr(caracter)} : {frecuencia}")

# Verificación
if texto == texto_decodificado:
    print("\nVerificación exitosa.")
else:
    print("\nError en decodificación.")


TABLA DE CÓDIGOS:
' ' : 000
'f' : 001
'a' : 010
'l' : 0110
'd' : 0111
'n' : 100
'm' : 101
'o' : 110
'h' : 1110
'u' : 1111

TEXTO CODIFICADO:
111011001100100001011111100011111000011101111001001101010100

TEXTO DECODIFICADO:
hola mundo huffman
FRECUENCIAS:
'h' : 2
'o' : 2
'l' : 1
'a' : 2
' ' : 2
'm' : 2
'u' : 2
'n' : 2
'd' : 1
'f' : 2

Verificación exitosa.
